In [ ]:
#@title 0. Install dependencies (run this cell once in Google Colab)
# Colab already provides a CUDA driver. CatBoost below detects it automatically.
!pip -q install --upgrade polars implicit lightgbm catboost h5py scikit-learn matplotlib shap mlflow tqdm ipywidgets


In [ ]:
from pathlib import Path
import subprocess

import numpy as np
import polars as pl
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    DATA_DIR = Path("/content/recsys_ranking")
else:
    SEMINAR_DIR = Path.cwd()
    if not (SEMINAR_DIR / "seminar.ipynb").exists():
        SEMINAR_DIR = Path.cwd() / "seminars" / "02_ranking"
    DATA_DIR = SEMINAR_DIR / "data"

DATA_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
METRIC_K = 10

try:
    subprocess.run(["nvidia-smi"], check=True, capture_output=True)
    USE_GPU = True
except (FileNotFoundError, PermissionError, subprocess.CalledProcessError):
    USE_GPU = False

# CatBoost GPU supports at most 1,023 documents in one query group.
# Keep the same limit for train and validation so model metrics stay comparable.
MAX_GROUP_SIZE = 1_023 if USE_GPU else 10_000

# The default makes the Colab demo fit reliably into a free GPU runtime.
# Set this to None to reproduce the experiment on the full Reddit matrix.
MAX_INTERACTIONS = 2_000_000 if IN_COLAB else None
NEGATIVE_PAIRS_PATH = DATA_DIR / f"negative_pairs_{MAX_INTERACTIONS or 'full'}.parquet"

LIGHTGBM_NUM_BOOST_ROUND = 100 if IN_COLAB else 300
CATBOOST_ITERATIONS = 300 if USE_GPU else 500
print(
    f"Running in {'Colab' if IN_COLAB else 'local Python'}; "
    f"CatBoost GPU: {USE_GPU}; maximum query size: {MAX_GROUP_SIZE}"
)


### 1. Load dataset

Today we'll work with Reddit dataset:
This dataset was released here:
https://www.reddit.com/r/redditdev/comments/dtg4j/want_to_help_reddit_build_a_recommender_a_public/

It contains 23M up/down votes from 44K users on 3.4M links.

We'll load it from `implicit` library.

In [ ]:
from implicit.datasets.reddit import get_reddit

# implicit stores the matrix as item × user; ranking groups must be users.
data = get_reddit().T.tocsr()


The downloaded matrix has items in rows and users in columns. Transpose it so that each ranking group is a user.


In [ ]:
def csr_to_triples(csr_mat):
    coo = csr_mat.tocoo()
    return np.vstack([coo.row, coo.col, coo.data]).T


def sample_negatives(interactions, num_negatives=5, seed=42, output_path=None):
    """Sample unseen popular items per user and optionally cache the result."""
    rng = np.random.default_rng(seed)
    popular_items = (
        interactions.group_by("item_id")
        .len()
        .sort("len", descending=True)
        .head(100_000)["item_id"]
        .to_numpy()
    )
    user_items = interactions.group_by("user_id").agg(
        pl.col("item_id").alias("seen_items")
    ).iter_rows(named=True)

    negatives = []
    for row in tqdm(user_items, desc="Sampling negatives"):
        candidates = popular_items[~np.isin(popular_items, row["seen_items"])]
        selected = rng.choice(candidates, size=min(num_negatives, len(candidates)), replace=False)
        negatives.extend((row["user_id"], item, 0) for item in selected)

    sampled = pl.DataFrame(
        negatives,
        schema={"user_id": pl.Int64, "item_id": pl.Int64, "target": pl.Int8},
        orient="row",
    )
    if output_path is not None:
        path = Path(output_path)
        path.parent.mkdir(parents=True, exist_ok=True)
        sampled.write_parquet(path)
    return sampled


triples = csr_to_triples(data)


In [ ]:
triples[:10]

In [ ]:
triples.shape

Today, we'll also use `polars` library instead of `pandas`. It works much faster.

Its syntax is very similar to **SQL** (Structured Query Language).

Documentation: https://docs.pola.rs/

In [ ]:
interactions_df = pl.DataFrame(
    triples,
    schema={"user_id": pl.Int64, "item_id": pl.Int64, "target": pl.Int8},
    orient="row",
)

if MAX_INTERACTIONS is not None and interactions_df.height > MAX_INTERACTIONS:
    interactions_df = interactions_df.sample(n=MAX_INTERACTIONS, seed=RANDOM_STATE)
    print(f"Using a reproducible sample of {interactions_df.height:,} interactions.")
else:
    print(f"Using all {interactions_df.height:,} interactions.")


In [ ]:
interactions_df


### 2. Preprocessing

In recommender systems, a **ranking task** focuses on ordering items such that the most relevant ones are presented at the top of the list for a given user. Unlike rating prediction, which estimates the explicit rating a user would give, **ranking** aims to sort items by **relevance**.

Given:
- A set of users $ U = \{u_1, u_2, ..., u_m\} $
- A set of items $ I = \{i_1, i_2, ..., i_n\} $
- Historical interaction data $ D = \{(u, i, y_{ui})\} $, where:
  - $ u $ is a user
  - $ i $ is an item
  - $ y_{ui} $ is a binary or implicit feedback signal (e.g., click, purchase, view)

Objective:
- Learn a scoring function $ f(u, i) $ that predicts the relevance of item $ i $ for user $ u $, such that for any user $ u $, the recommended list of items $ i_1, ..., i_k $ is ranked in descending order of predicted relevance:
  
  $$ f(u, i_1) \geq f(u, i_2) \geq \dots \geq f(u, i_k) $$

### How to read ranking metrics

Classification accuracy does not answer the ranking question: it treats a relevant item at rank 1 and rank 100 equally. Therefore all metrics below are computed **inside each user group** and then averaged across users, at a fixed cutoff `K`.

- **NDCG@K** rewards relevant items near the top: $DCG@K = \sum_{r=1}^{K}\frac{2^{rel_r}-1}{\log_2(r+1)}$, normalized by the ideal ordering. It is the best general-purpose metric here because it accounts for several relevant objects and their positions.
- **MAP@K** averages precision at every rank where a relevant item appears. Use it when the quality of the whole relevant set matters.
- **MRR@K** is $1/r$ for the first relevant item at rank $r$ (or 0 if it is absent from the top K). It is appropriate when finding one good answer quickly is the main goal.

The metrics are only meaningful for the same candidate-generation policy, user split, relevance definition, and cutoff. In this seminar: relevance is an upvote, the split is by user, negatives are sampled unseen popular items, and `K=10`.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

# One recommendation list for one user. The metrics below use the same binary relevance.
relevance_example = np.array([0, 0, 1, 0, 1, 0, 0, 1, 0, 0])
ranks = np.arange(1, len(relevance_example) + 1)
precision_at_relevant = np.cumsum(relevance_example) / ranks
ap_at_10 = precision_at_relevant[relevance_example == 1].mean()
dcg_at_10 = np.sum(relevance_example / np.log2(ranks + 1))
ideal_dcg_at_10 = np.sum(np.sort(relevance_example)[::-1] / np.log2(ranks + 1))
ndcg_at_10 = dcg_at_10 / ideal_dcg_at_10
first_relevant_rank = ranks[np.flatnonzero(relevance_example)[0]]
mrr_at_10 = 1 / first_relevant_rank

fig, (rank_ax, metric_ax) = plt.subplots(
    2,
    1,
    figsize=(12, 4.8),
    height_ratios=[1.25, 0.85],
    layout="constrained",
)

rank_ax.set_title(
    "One user's recommendation list: earlier relevant items are better",
    loc="left",
    fontweight="bold",
)
for rank, relevance in zip(ranks, relevance_example):
    facecolor = "#2a9d8f" if relevance else "#e9ecef"
    box = FancyBboxPatch(
        (rank - 0.38, 0.22),
        0.76,
        0.48,
        boxstyle="round,pad=0.02,rounding_size=0.04",
        facecolor=facecolor,
        edgecolor="none",
    )
    rank_ax.add_patch(box)
    rank_ax.text(rank, 0.87, f"rank {rank}", ha="center", va="center", fontsize=9)
    rank_ax.text(
        rank,
        0.46,
        "relevant" if relevance else "not relevant",
        ha="center",
        va="center",
        fontsize=8,
        color="white" if relevance else "#495057",
    )
rank_ax.text(
    0.02,
    0.02,
    "Green tiles are relevant; gray tiles are not relevant.",
    transform=rank_ax.transAxes,
    color="#495057",
)
rank_ax.set(xlim=(0.35, 10.65), ylim=(0, 1.08))
rank_ax.axis("off")

metric_ax.axis("off")
metric_cards = [
    ("NDCG@10", f"{ndcg_at_10:.3f}", "Rewards relevance near the top."),
    ("MAP@10", f"{ap_at_10:.3f}", "Rewards finding all relevant items."),
    ("MRR@10", f"{mrr_at_10:.3f}", f"First relevant item: rank {first_relevant_rank}."),
]
for index, (name, value, explanation) in enumerate(metric_cards):
    x = (index + 0.5) / len(metric_cards)
    metric_ax.text(x, 0.82, name, ha="center", va="center", fontsize=11, fontweight="bold")
    metric_ax.text(x, 0.50, value, ha="center", va="center", fontsize=22, color="#2a9d8f")
    metric_ax.text(x, 0.17, explanation, ha="center", va="center", fontsize=10, color="#495057")
    if index < len(metric_cards) - 1:
        metric_ax.axvline((index + 1) / len(metric_cards), ymin=0.12, ymax=0.88, color="#dee2e6")

plt.show()


It means that we need to have not only positive examples but also **negatives**.

In our dataset we have up/down votes:
* 1 - upvote
* -1 - downvote
* 0 - Seen but no interaction (neutral or skipped)

Let's look at distribution of it:

In [ ]:
interactions_df.group_by("target").agg(pl.len())


The negative pairs are generated deterministically from popular, unseen items and cached locally in `data/negative_pairs.parquet`. The cache is ignored by Git; the first run can take several minutes.


In [ ]:
if not NEGATIVE_PAIRS_PATH.exists():
    sample_negatives(
        interactions_df,
        num_negatives=5,
        seed=RANDOM_STATE,
        output_path=NEGATIVE_PAIRS_PATH,
    )

negatives_df = pl.read_parquet(NEGATIVE_PAIRS_PATH)


In [ ]:
negatives_df.head()

After that, we can concat our dataframes for full dataset:

In [ ]:
pairs = pl.concat([interactions_df, negatives_df]).with_columns(
    (pl.col("target") == 1).cast(pl.Int8).alias("relevance"),
    (pl.col("target") == -1).cast(pl.Int8).alias("is_downvote"),
)


In [ ]:
pairs


ALS -> R ~ XY, X - user matrix, Y - item matrix, R (m x n), X (m x k), Y (k x n)

For each pair (u, i):
1) dot_product(x_u, y_i)
2) cosine_sim(x_u, y_i)
3) cosine_dist(x_u, y_i) (1 - cosine_sim)

### 3. Feature engineering

For ranking task we can generate different features. Stop here and think: what features can be suitable for it?

To avoid target leakage, fit all counters on training users only. For a training row, subtract its own reaction from its counters; validation rows use only statistics available from training users.


In [ ]:
all_users = pairs["user_id"].unique().to_numpy()
train_users, valid_users = train_test_split(
    all_users,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

train_pairs = pairs.filter(pl.col("user_id").is_in(train_users))
valid_pairs = pairs.filter(pl.col("user_id").is_in(valid_users))


Then we can join these features with our ranking pairs:

In [ ]:
FEATURE_COLS = [
    "user_counter_upvote",
    "user_counter_downvote",
    "item_counter_upvote",
    "item_counter_downvote",
]


def fit_counter_stats(train_pairs):
    user_stats = train_pairs.group_by("user_id").agg(
        pl.col("relevance").sum().alias("user_counter_upvote"),
        pl.col("is_downvote").sum().alias("user_counter_downvote"),
    )
    item_stats = train_pairs.group_by("item_id").agg(
        pl.col("relevance").sum().alias("item_counter_upvote"),
        pl.col("is_downvote").sum().alias("item_counter_downvote"),
    )
    return user_stats, item_stats


def add_counter_features(pairs, user_stats, item_stats, training):
    featured = pairs.join(user_stats, on="user_id", how="left")
    featured = featured.join(item_stats, on="item_id", how="left")
    if training:
        featured = featured.with_columns(
            (pl.col("user_counter_upvote") - pl.col("relevance")).alias("user_counter_upvote"),
            (pl.col("user_counter_downvote") - pl.col("is_downvote")).alias("user_counter_downvote"),
            (pl.col("item_counter_upvote") - pl.col("relevance")).alias("item_counter_upvote"),
            (pl.col("item_counter_downvote") - pl.col("is_downvote")).alias("item_counter_downvote"),
        )
    return featured.with_columns(
        pl.col(FEATURE_COLS).fill_null(0).cast(pl.Int32)
    )


user_stats, item_stats = fit_counter_stats(train_pairs)


For now we have in our dataset three target values: -1, 1, 0. Let's make target binary: we need to convert -1 to 0 as negative pairs.

In [ ]:
train_df = add_counter_features(train_pairs, user_stats, item_stats, training=True)
valid_df = add_counter_features(valid_pairs, user_stats, item_stats, training=False)


In [ ]:
train_df


### 4. LightGBM

### 4. LightGBM

The standard `pip install lightgbm` wheel used by Colab runs on CPU. It remains useful as a fast baseline; the CatBoost experiment below automatically uses the attached Colab GPU. The default Colab sample and iteration budgets are deliberately modest, and can be increased after the first successful run.


In [ ]:
import lightgbm as lgb


In [ ]:
def prepare_rank_data(df):
    group_sizes = df.group_by("user_id").agg(pl.len().alias("group_size"))
    eligible_users = group_sizes.filter(pl.col("group_size") <= MAX_GROUP_SIZE)
    excluded_count = group_sizes.height - eligible_users.height
    if excluded_count:
        print(
            f"Excluded {excluded_count} groups larger than {MAX_GROUP_SIZE} objects "
            f"to satisfy the ranking-model limit."
        )

    ordered = (
        df.join(eligible_users.select("user_id"), on="user_id", how="semi")
        .sort(["user_id", "item_id"])
    )
    groups = (
        ordered.group_by("user_id")
        .agg(pl.len().alias("group_size"))
        .sort("user_id")["group_size"]
        .to_numpy()
    )
    assert groups.sum() == len(ordered)
    assert groups.max() <= MAX_GROUP_SIZE
    return ordered, groups


train_rank_df, train_groups = prepare_rank_data(train_df)
valid_rank_df, valid_groups = prepare_rank_data(valid_df)


In [ ]:
X_train = train_rank_df.select(FEATURE_COLS).to_numpy()
y_train = train_rank_df["relevance"].to_numpy()
X_valid = valid_rank_df.select(FEATURE_COLS).to_numpy()
y_valid = valid_rank_df["relevance"].to_numpy()

train_data = lgb.Dataset(X_train, label=y_train, group=train_groups)
valid_data = lgb.Dataset(X_valid, label=y_valid, group=valid_groups, reference=train_data)


Then we can select needed parameters to train our model:

In [ ]:
params = {
    "objective": "lambdarank",
    "metric": ["ndcg", "map"],
    "eval_at": [METRIC_K],
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 20,
    "lambda_l1": 0.1,
    "lambda_l2": 0.1,
    "seed": RANDOM_STATE,
    "verbosity": -1,
    # The standard pip wheel is CPU-only. CatBoost below uses the Colab GPU.
    "device_type": "cpu",
}

lightgbm_model = lgb.train(
    params,
    train_data,
    num_boost_round=LIGHTGBM_NUM_BOOST_ROUND,
    valid_sets=[valid_data],
    valid_names=["valid"],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)],
)


Metrics are reported at a fixed cutoff, `K=10`. The split is by user, so this evaluates ranking for unseen users. When GPU is active, groups larger than 1,023 objects are excluded before both models are trained: this is a CatBoost GPU limit, not a metric cutoff.


### 5. CatBoost

Let's make the same for CatBoost.

Documentation: https://catboost.ai/docs/en/concepts/loss-functions-ranking

Example: https://github.com/catboost/catboost/blob/master/catboost/tutorials/ranking/ranking_tutorial.ipynb

In [ ]:
from catboost import CatBoostRanker, Pool


def make_pool(df):
    return Pool(
        data=df.select(FEATURE_COLS).to_numpy(),
        label=df["relevance"].to_numpy(),
        group_id=df["user_id"].to_numpy(),
    )


train_pool = make_pool(train_rank_df)
valid_pool = make_pool(valid_rank_df)


In [ ]:
catboost_params = {
    "iterations": CATBOOST_ITERATIONS,
    "learning_rate": 0.1,
    "depth": 6,
    "loss_function": "YetiRank",
    "eval_metric": f"NDCG:top={METRIC_K}",
    "custom_metric": [f"MAP:top={METRIC_K}", f"MRR:top={METRIC_K}"],
    "verbose": 100,
    "random_seed": RANDOM_STATE,
}
if USE_GPU:
    catboost_params.update(task_type="GPU", devices="0")
else:
    catboost_params.update(task_type="CPU")

catboost_model = CatBoostRanker(**catboost_params)


In [ ]:
catboost_model.fit(
    train_pool,
    eval_set=valid_pool,
    use_best_model=True,
    early_stopping_rounds=50,
)


In [ ]:
import matplotlib.pyplot as plt

results = catboost_model.get_evals_result()
validation_metrics = results["validation"]
fig, axs = plt.subplots(len(validation_metrics), 1, figsize=(10, 3 * len(validation_metrics)))

for ax, (metric_name, values) in zip(np.atleast_1d(axs), validation_metrics.items()):
    if metric_name in results["learn"]:
        ax.plot(results["learn"][metric_name], label="train")
    ax.plot(values, label="validation")
    ax.set_title(metric_name)
    ax.grid()
    ax.legend()

plt.tight_layout()


In [ ]:
feature_importance = catboost_model.get_feature_importance(type="PredictionValuesChange")
importance_df = pl.DataFrame(
    {"feature": FEATURE_COLS, "importance": feature_importance}
).sort("importance", descending=True)
importance_df


In [ ]:
RUN_SHAP = False

if RUN_SHAP:
    import shap

    shap_values = catboost_model.get_feature_importance(
        valid_pool,
        type="ShapValues",
        verbose=100,
    )[:, :-1]
    shap.summary_plot(
        shap_values,
        valid_pool.get_features(),
        feature_names=FEATURE_COLS,
        plot_type="bar",
    )


### 6*. Logging CatBoost with MLFlow

Documentation: 
https://mlflow.org/docs/latest/api_reference/python_api/mlflow.catboost.html

In [ ]:
RUN_MLFLOW = False

if RUN_MLFLOW:
    import mlflow
    import mlflow.catboost

    mlflow.set_tracking_uri((DATA_DIR / "mlruns").resolve().as_uri())
    mlflow.set_experiment("catboost_ranking")

    with mlflow.start_run():
        mlflow.log_params(catboost_model.get_params())
        mlflow.log_metrics({name: values[-1] for name, values in validation_metrics.items()})
        mlflow.catboost.log_model(catboost_model, name="model")
        mlflow.log_figure(plt.gcf(), "feature_importance.png")
